# Session 23 — Pandas analysis
## Follow-along

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ffraile/computer_science_tutorials/blob/main/source/Modularization/follow_along/Session%2023%20-%20Pandas%20analysis%20-%20follow%20along.ipynb) [![Github](https://img.shields.io/badge/download-github-blue?logo=github)](https://github.com/ffraile/computer_science_tutorials/raw/main/source/Modularization/follow_along/Session%2023%20-%20Pandas%20analysis%20-%20follow%20along.ipynb)


## Learning goals

- define table grain, keys and expected cardinality;
- perform and compare `inner` and `left` joins;
- detect unmatched records and duplicated lookup keys;
- validate a many-to-one relationship;
- calculate derived values and sort records;
- group and aggregate by one or more variables;
- reconcile totals and visualise an aggregate result.


## Working routine

> **predict → execute → validate → interpret**

Before every join, state the expected relationship and row count. Before every aggregation, state the result grain and a total that should be preserved.


## Use of AI assistance

Copy the complete prompt below into a new Gemini conversation. Include your predicted grain, key, row count or total when requesting help. You remain responsible for verifying every suggestion.

```text
You are a coding tutor for an introductory Computer Science course. Help me reason and work independently; do not complete my work for me.

TUTORING RULES
- Begin by asking for my goal, prediction, current attempt or observed evidence.
- Ask one focused question at a time and wait for my answer.
- Use the lowest useful help level: clarify → one conceptual hint → steps or incomplete pseudocode → a small incomplete code fragment → review code I wrote.
- Never provide a complete exercise solution or fill an answer cell unless the teacher explicitly enables COMPLETE_SOLUTION.
- Preserve notebook statements, tests and existing code. Do not complete future cells.
- Follow predict → execute → validate → interpret.
- When debugging, use evidence, one hypothesis, one diagnostic check and the smallest correction.
- Keep each response concise and wait after one question or next step.

ACTIVITY CONTEXT
- Session: 23 — Pandas analysis
- Activity type: FOLLOW-ALONG
- Students already know: DataFrames and Series; CSV loading; inspection; loc and iloc; Boolean filters; missing values; type conversion; duplicates; justified cleaning; NumPy and Matplotlib
- This activity introduces: table grain; keys and cardinality; merge with inner and left joins; unmatched records; duplicate lookup keys; indicator and validate; derived columns; sort_values; groupby; count, sum, mean, min and max; named aggregation; total reconciliation; and simple aggregate visualisation
- Do not use: right or outer joins in depth; concatenation; joins on indexes; MultiIndex; pivot tables; apply(); lambda functions; rolling operations; advanced plotting; classes; recursion; generators; or decorators
- Maximum help: PARTIAL_CODE
```


## Scenario — Orders and product catalogue

In this scenario we are going to manage data of a website that sells electronic components to build Arduino prototypes. We focus on two tables, `orders`, and `products`. Recall that Tables organise data into **rows** or **records**. For instance, each row in the `orders` table describes an order that a customer placed on the website, and each row in the `products` table describes a product that is available for sale. Each row is organised into **columns** or **fields**, which describe the attributes of the record. Let us look at the two tables:
- The `orders` table records transactions. The fields are:
  - `order_id` — a **unique identifier** for each order;
  - `product_id` — a **unique identifier** for each product;
  - `quantity` — the number of units ordered.
- The `products` table supplies names, categories and prices:
  - `product_id` — a **unique identifier** for each product;
  - `product_name` — the name of the product;
  - `category` — the category to which the product belongs;
  - `unit_price` — the price per unit of the product.

Notice that we introduce the concept of a **unique identifier**. A unique identifier is a field that uniquely identifies each record in a table. In the `orders` table, the `order_id` field is a unique identifier, and in the `products` table, the `product_id` field is a unique identifier. We will use these unique identifiers to join the two tables together.

To create a table with Pandas, we can use the `pd.DataFrame()` constructor and pass a list of dictionaries, where each dictionary represents a row in the table. Each key in the dictionary corresponds to a column name, and the value corresponds to the value for that column in that row. For example, to create the `orders` table, we can use the following code:


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

orders = pd.DataFrame([
    {"order_id":"O-101", "product_id":"P-01", "quantity":2},
    {"order_id":"O-102", "product_id":"P-02", "quantity":1},
    {"order_id":"O-103", "product_id":"P-01", "quantity":3},
    {"order_id":"O-104", "product_id":"P-03", "quantity":4},
    {"order_id":"O-105", "product_id":"P-99", "quantity":1},
])

products = pd.DataFrame([
    {"product_id":"P-01", "product_name":"Sensor kit", "category":"Electronics", "unit_price":25.0},
    {"product_id":"P-02", "product_name":"Cable pack", "category":"Accessories", "unit_price":8.0},
    {"product_id":"P-03", "product_name":"Mounting set", "category":"Accessories", "unit_price":12.0},
])


## 1. Define keys and cardinality

The **key** is a table field that uniquely identifies each record. We expect that **keys** are unique and non-missing, hence the field name `product_id` is a candidate key in the `products` table, and `order_id` is a candidate key in the `orders` table.

| Table | One row represents | Candidate key |
|---|---|---|
| `orders` | one order line | `order_id` |
| `products` | one catalogue product | `product_id` |


The **cardinality** is the number of records in one table that can match a record in another table. For instance, we expect that many order lines can match one product. This is a **many-to-one** relationship. Other relationships are **one-to-one**, **one-to-many** and **many-to-many**.


### Keys, references and grain

Notice that `product_id` also appears in the `orders` table, but there it is not unique: orders `O-101` and `O-103` both refer to product `P-01`. In `orders`, `product_id` is a **reference**, also called a **foreign key**: a field whose values should match the key of another table. This shared field is what allows us to connect, or **join**, the two tables: from each order line, we can find the product it refers to.

We also describe the **grain** of each table, that is, what one row represents. In `orders`, one row represents one order line; in `products`, one row represents one catalogue product. Keeping the grain in mind helps us predict how many rows an operation should produce, and detect when a result has too many or too few rows.

Before joining, we check our assumptions with code:

- `is_unique` is a Series attribute that is `True` when no value is repeated;
- `isna()` returns a mask that is `True` for missing values, and `sum()` counts them.

In [ ]:
print("Order rows:", len(orders))
print("Unique order IDs?", orders["order_id"].is_unique)
print("Unique catalogue product IDs?", products["product_id"].is_unique)
print("Missing order product IDs:", orders["product_id"].isna().sum())


## 2. Compare inner and left joins

A **join** combines two tables into one by matching rows that share the same value in a key field. In Pandas, the DataFrame method `merge()` performs joins. The expression `orders.merge(products, on="product_id")` takes every row of `orders`, looks for the row of `products` with the same `product_id`, and builds a new row that contains the columns of both tables.

The table on which we call the method (`orders`) is the **left** table, and the table passed as argument (`products`) is the **right** table. The parameter `on` names the field used to match rows, and the parameter `how` decides what happens to rows without a match:

| `how` | Rows kept | Left rows without a match |
|---|---|---|
| `"inner"` | Only rows with a match in both tables | Removed |
| `"left"` | Every row of the left table | Kept, with missing values (`NaN`) in the columns of the right table |

The argument `indicator=True` adds a column called `_merge` that records where each result row came from.

Look at the data again: order `O-105` refers to product `P-99`, which does not exist in the catalogue. Predict the row count and the fate of `O-105` in each result.

In [ ]:
inner_result = orders.merge(products, on="product_id", how="inner")
left_result = orders.merge(products, on="product_id", how="left", indicator=True)

print("Inner rows:", len(inner_result))
print("Left rows:", len(left_result))
display(left_result)


## 3. Audit unmatched records

An **audit** is a check that looks for records that do not behave as expected, so that they can be reported and investigated. The merge indicator records one of three values:

- `both` — the row found a match in both tables;
- `left_only` — the row exists only in the left table;
- `right_only` — the row exists only in the right table (this value cannot appear in a left join).

In a left join, `left_only` identifies order records with no catalogue match. The Series method `unique()` returns each distinct value once, which gives a short list of the unknown product identifiers.

In [ ]:
unmatched_orders = left_result.loc[left_result["_merge"] == "left_only"]
display(unmatched_orders)
print("Unmatched product IDs:", unmatched_orders["product_id"].unique())


## 4. Validate the relationship

In section 1 we expected a **many-to-one** relationship: many order lines can refer to the same product, but each product identifier must appear only once in the catalogue. The argument `validate="many_to_one"` turns this assumption into an executable check: `merge()` verifies that the key is unique in the right table and raises a `MergeError` if it is not.

The `assert` after the join checks the grain: a left join with a many-to-one relationship must produce exactly one row per order line.

In [ ]:
enriched = orders.merge(
    products,
    on="product_id",
    how="left",
    indicator=True,
    validate="many_to_one",
)

assert len(enriched) == len(orders)


### Why duplicates are dangerous

If `product_id` appeared twice in the catalogue, matching order rows would multiply. The join might execute, but the result would no longer have one row per order line. `validate` prevents that silent change.


## 5. Separate matched records for sales analysis

The unknown product has no trusted price. Keep it in the audit output, but exclude it from a calculation that requires catalogue information.

- A Boolean mask on `_merge` keeps only the rows with a match in both tables. As in session 22, `copy()` makes the new table independent.
- `drop(columns="_merge")` returns the table without the indicator column, which we no longer need.
- The last line creates a **derived column**: a new column calculated from other columns. Assigning to a column label that does not exist adds a new column, as with dictionary keys. The multiplication is element-wise, so each row gets its own `quantity × unit_price`.

In [ ]:
matched_orders = enriched.loc[enriched["_merge"] == "both"].copy()
matched_orders = matched_orders.drop(columns="_merge")
matched_orders["sales"] = matched_orders["quantity"] * matched_orders["unit_price"]

display(matched_orders)


## 6. Sort records

`sort_values()` returns a new table with the rows reordered according to one or more columns, without changing any value. Each row keeps its index label.

- The first argument is the list of columns used to sort. Rows are ordered by `sales`; when two rows have the same sales, the second column, `order_id`, decides their order. A column used this way is called a **tie-breaker**. It makes the result **deterministic**: it is always the same, whatever the original order of the rows.
- `ascending` gives one Boolean per column: `False` means from the largest to the smallest value, and `True` from the smallest to the largest.

In [ ]:
ranked_orders = matched_orders.sort_values(
    ["sales", "order_id"],
    ascending=[False, True],
)
display(ranked_orders)


## 7. Group and aggregate

To summarise sales by category, we follow three steps:

1. **group**: split the rows into groups that share the same value in a column, here `category`;
2. **aggregate**: reduce each group to a few summary values, such as a count, a sum or a mean. A function that turns many values into one value is an **aggregation function**;
3. **combine**: build a new table with one row per group.

In Pandas, `groupby("category")` performs the grouping, and `agg()` defines the aggregations. With **named aggregation**, each argument has the form `new_column=("source_column", "function")`. For instance, `total_sales=("sales", "sum")` creates a column called `total_sales` with the sum of `sales` in each group. The functions we use are `count`, `sum`, `mean`, `min` and `max`.

The argument `as_index=False` keeps `category` as an ordinary column instead of using it as the index of the result.

Notice the parentheses around the whole expression. They let us write one method call per line: each method is applied to the result of the previous one. This style is called **method chaining**.

The grain changes: before, one row represented one order line; after, one row represents one category. Predict one result row per category.

In [ ]:
category_summary = (
    matched_orders
    .groupby("category", as_index=False)
    .agg(
        order_lines=("order_id", "count"),
        total_units=("quantity", "sum"),
        total_sales=("sales", "sum"),
        mean_unit_price=("unit_price", "mean"),
        min_unit_price=("unit_price", "min"),
        max_unit_price=("unit_price", "max"),
    )
    .sort_values("total_sales", ascending=False)
)

display(category_summary)


## 8. Reconcile totals

**Reconciliation** means checking that a total calculated in two different ways gives the same result. Aggregation should reorganise the data, not create or lose it: the sum of category sales must equal the sum of the sales of the included order lines. Counts and units can be reconciled in the same way.

In [ ]:
assert category_summary["total_sales"].sum() == matched_orders["sales"].sum()
assert category_summary["total_units"].sum() == matched_orders["quantity"].sum()
assert category_summary["order_lines"].sum() == len(matched_orders)

print("Reconciled sales:", category_summary["total_sales"].sum())


## 9. Group by two variables

Passing a list of columns to `groupby()` creates one group for each combination of values that appears in the data. Adding a second grouping field changes the result grain to one row per category and product.

Predict the number of rows before running the cell.

In [ ]:
product_summary = (
    matched_orders
    .groupby(["category", "product_name"], as_index=False)
    .agg(total_units=("quantity", "sum"), total_sales=("sales", "sum"))
)
display(product_summary)


## 10. Visualise and interpret

DataFrames can create Matplotlib charts directly. `category_summary.plot.bar(x="category", y="total_sales")` draws a bar chart with one bar per row: the column `x` gives the labels of the bars and the column `y` gives their heights. `legend=False` hides the legend, since there is only one series. Because the chart is drawn with Matplotlib, we can still use `plt.ylabel()`, `plt.title()` and `plt.show()`. `plt.tight_layout()` adjusts the margins so that labels are not cut.

The chart communicates one chosen aggregation. It does not show unmatched orders and does not validate the join.

In [ ]:
category_summary.plot.bar(x="category", y="total_sales", legend=False)
plt.ylabel("Sales")
plt.title("Sales by product category")
plt.tight_layout()
plt.show()


## Final checkpoint

1. What does one row represent before and after the join?
2. Why was a left join useful for auditing?
3. What does `validate="many_to_one"` protect?
4. Why was the unknown product excluded from sales but retained in the audit?
5. What is the result grain of each summary?
6. Which reconciliation checks demonstrate that aggregation preserved included records?
